# TP 1 : Gestion de stock de pharmacie (Spring Boot + MySQL)

**Module :** Spring Boot - Microservices  **Prof :** Youness IDRISSI KHAMLICHI  **ENSA Fès**


## 1. Structure du projet

In [1]:
import os
dossiers = [
    "pharmacie/src/main/java/ma/ensa/pharmacie/entity",
    "pharmacie/src/main/java/ma/ensa/pharmacie/repository",
    "pharmacie/src/main/java/ma/ensa/pharmacie/service",
    "pharmacie/src/main/java/ma/ensa/pharmacie/controller",
    "pharmacie/src/main/resources/templates",
    "postman",
]
for d in dossiers:
    os.makedirs(d, exist_ok=True)
print("Dossiers créés :", len(dossiers))

Dossiers créés : 6


## 2. `pom.xml`
Dépendances : Web, Data JPA, Thymeleaf, driver MySQL et Lombok (pour éviter d'écrire tous les getters/setters).

In [2]:
%%writefile pharmacie/pom.xml
<?xml version="1.0" encoding="UTF-8"?>
<project xmlns="http://maven.apache.org/POM/4.0.0"
         xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
         xsi:schemaLocation="http://maven.apache.org/POM/4.0.0 https://maven.apache.org/xsd/maven-4.0.0.xsd">
    <modelVersion>4.0.0</modelVersion>

    <parent>
        <groupId>org.springframework.boot</groupId>
        <artifactId>spring-boot-starter-parent</artifactId>
        <version>3.3.5</version>
        <relativePath/>
    </parent>

    <groupId>ma.ensa</groupId>
    <artifactId>pharmacie</artifactId>
    <version>0.0.1-SNAPSHOT</version>
    <name>pharmacie</name>

    <properties>
        <java.version>17</java.version>
    </properties>

    <dependencies>
        <dependency>
            <groupId>org.springframework.boot</groupId>
            <artifactId>spring-boot-starter-web</artifactId>
        </dependency>
        <dependency>
            <groupId>org.springframework.boot</groupId>
            <artifactId>spring-boot-starter-data-jpa</artifactId>
        </dependency>
        <dependency>
            <groupId>org.springframework.boot</groupId>
            <artifactId>spring-boot-starter-thymeleaf</artifactId>
        </dependency>
        <dependency>
            <groupId>com.mysql</groupId>
            <artifactId>mysql-connector-j</artifactId>
            <scope>runtime</scope>
        </dependency>
        <dependency>
            <groupId>org.projectlombok</groupId>
            <artifactId>lombok</artifactId>
            <optional>true</optional>
        </dependency>
    </dependencies>

    <build>
        <plugins>
            <plugin>
                <groupId>org.springframework.boot</groupId>
                <artifactId>spring-boot-maven-plugin</artifactId>
                <configuration>
                    <excludes>
                        <exclude>
                            <groupId>org.projectlombok</groupId>
                            <artifactId>lombok</artifactId>
                        </exclude>
                    </excludes>
                </configuration>
            </plugin>
        </plugins>
    </build>
</project>

Overwriting pharmacie/pom.xml


## 3. Classe principale

In [3]:
%%writefile pharmacie/src/main/java/ma/ensa/pharmacie/PharmacieApplication.java
package ma.ensa.pharmacie;

import org.springframework.boot.SpringApplication;
import org.springframework.boot.autoconfigure.SpringBootApplication;

@SpringBootApplication
public class PharmacieApplication {

    public static void main(String[] args) {
        SpringApplication.run(PharmacieApplication.class, args);
    }
}

Overwriting pharmacie/src/main/java/ma/ensa/pharmacie/PharmacieApplication.java


## 4. Configuration MySQL (`application.properties`)

- `createDatabaseIfNotExist=true` : la base `pharmacie_db` est créée toute seule.
- `ddl-auto=update` : Hibernate crée les tables à partir des entités.
- `defer-datasource-initialization=true` : `data.sql` s'exécute **après** la création des tables par Hibernate.

In [4]:
%%writefile pharmacie/src/main/resources/application.properties
spring.application.name=pharmacie

# --- MySQL ---
spring.datasource.url=jdbc:mysql://localhost:3306/pharmacie_db?createDatabaseIfNotExist=true&useSSL=false&allowPublicKeyRetrieval=true&serverTimezone=UTC
spring.datasource.username=root
spring.datasource.password=

# --- JPA / Hibernate ---
spring.jpa.hibernate.ddl-auto=update
spring.jpa.show-sql=true
spring.jpa.defer-datasource-initialization=true

# --- Jeu de données (data.sql) ---
spring.sql.init.mode=always
spring.sql.init.encoding=UTF-8

server.port=8080

Overwriting pharmacie/src/main/resources/application.properties


## 5. Entités JPA

Les relations du diagramme de classe :

| Relation | Annotation | Où |
|---|---|---|
| Categorie 1 → * Produit | `@OneToMany` / `@ManyToOne` | `Categorie.produits` / `Produit.categorie` |
| Fournisseur 1 → * Produit | `@OneToMany` / `@ManyToOne` | `Fournisseur.produits` / `Produit.fournisseur` |
| Produit 1 → 1 DetailProduit | `@OneToOne` (cascade) | `Produit.detailProduit` (clé `detail_id`) |
| Produit * ↔ * Tag | `@ManyToMany` + table `produit_tag` | `Produit.tags` |

Petit piège : les listes `produits` côté `Categorie`, `Fournisseur` et `Tag` sont en `@JsonIgnore`, sinon la sérialisation JSON boucle à l'infini (Produit → Categorie → Produit → ...).

In [5]:
%%writefile pharmacie/src/main/java/ma/ensa/pharmacie/entity/Categorie.java
package ma.ensa.pharmacie.entity;

import com.fasterxml.jackson.annotation.JsonIgnore;
import jakarta.persistence.*;
import lombok.Getter;
import lombok.NoArgsConstructor;
import lombok.Setter;

import java.util.ArrayList;
import java.util.List;

@Entity
@Getter
@Setter
@NoArgsConstructor
public class Categorie {

    @Id
    @GeneratedValue(strategy = GenerationType.IDENTITY)
    private Long id;

    private String nom;

    @Column(length = 500)
    private String description;

    // une catégorie contient plusieurs produits
    @OneToMany(mappedBy = "categorie")
    @JsonIgnore
    private List<Produit> produits = new ArrayList<>();
}

Overwriting pharmacie/src/main/java/ma/ensa/pharmacie/entity/Categorie.java


In [6]:
%%writefile pharmacie/src/main/java/ma/ensa/pharmacie/entity/Fournisseur.java
package ma.ensa.pharmacie.entity;

import com.fasterxml.jackson.annotation.JsonIgnore;
import jakarta.persistence.*;
import lombok.Getter;
import lombok.NoArgsConstructor;
import lombok.Setter;

import java.util.ArrayList;
import java.util.List;

@Entity
@Getter
@Setter
@NoArgsConstructor
public class Fournisseur {

    @Id
    @GeneratedValue(strategy = GenerationType.IDENTITY)
    private Long id;

    private String nom;
    private String contact;
    private String adresse;

    @OneToMany(mappedBy = "fournisseur")
    @JsonIgnore
    private List<Produit> produits = new ArrayList<>();
}

Overwriting pharmacie/src/main/java/ma/ensa/pharmacie/entity/Fournisseur.java


In [7]:
%%writefile pharmacie/src/main/java/ma/ensa/pharmacie/entity/DetailProduit.java
package ma.ensa.pharmacie.entity;

import jakarta.persistence.*;
import lombok.Getter;
import lombok.NoArgsConstructor;
import lombok.Setter;

@Entity
@Table(name = "detail_produit")
@Getter
@Setter
@NoArgsConstructor
public class DetailProduit {

    @Id
    @GeneratedValue(strategy = GenerationType.IDENTITY)
    private Long id;

    private String fabricant;
    private String paysOrigine;
    private String composition;
    private String dosage;

    @Column(length = 500)
    private String indications;
}

Overwriting pharmacie/src/main/java/ma/ensa/pharmacie/entity/DetailProduit.java


In [8]:
%%writefile pharmacie/src/main/java/ma/ensa/pharmacie/entity/Tag.java
package ma.ensa.pharmacie.entity;

import com.fasterxml.jackson.annotation.JsonIgnore;
import jakarta.persistence.*;
import lombok.Getter;
import lombok.NoArgsConstructor;
import lombok.Setter;

import java.util.HashSet;
import java.util.Set;

@Entity
@Getter
@Setter
@NoArgsConstructor
public class Tag {

    @Id
    @GeneratedValue(strategy = GenerationType.IDENTITY)
    private Long id;

    private String libelle;

    // côté inverse de la relation ManyToMany (le propriétaire est Produit)
    @ManyToMany(mappedBy = "tags")
    @JsonIgnore
    private Set<Produit> produits = new HashSet<>();
}

Overwriting pharmacie/src/main/java/ma/ensa/pharmacie/entity/Tag.java


In [9]:
%%writefile pharmacie/src/main/java/ma/ensa/pharmacie/entity/Produit.java
package ma.ensa.pharmacie.entity;

import jakarta.persistence.*;
import lombok.Getter;
import lombok.NoArgsConstructor;
import lombok.Setter;

import java.math.BigDecimal;
import java.time.LocalDate;
import java.util.HashSet;
import java.util.Set;

@Entity
@Getter
@Setter
@NoArgsConstructor
public class Produit {

    @Id
    @GeneratedValue(strategy = GenerationType.IDENTITY)
    private Long id;

    private String nom;

    @Column(length = 500)
    private String description;

    private BigDecimal prix;
    private String codeBarre;
    private LocalDate dateExpiration;

    @ManyToOne
    @JoinColumn(name = "categorie_id")
    private Categorie categorie;

    @ManyToOne
    @JoinColumn(name = "fournisseur_id")
    private Fournisseur fournisseur;

    // le détail est créé / supprimé avec le produit
    @OneToOne(cascade = CascadeType.ALL, orphanRemoval = true)
    @JoinColumn(name = "detail_id")
    private DetailProduit detailProduit;

    @ManyToMany
    @JoinTable(
            name = "produit_tag",
            joinColumns = @JoinColumn(name = "produit_id"),
            inverseJoinColumns = @JoinColumn(name = "tag_id")
    )
    private Set<Tag> tags = new HashSet<>();
}

Overwriting pharmacie/src/main/java/ma/ensa/pharmacie/entity/Produit.java


## 6. Repositories (`JpaRepository`)

In [10]:
%%writefile pharmacie/src/main/java/ma/ensa/pharmacie/repository/CategorieRepository.java
package ma.ensa.pharmacie.repository;

import ma.ensa.pharmacie.entity.Categorie;
import org.springframework.data.jpa.repository.JpaRepository;

public interface CategorieRepository extends JpaRepository<Categorie, Long> {
}

Overwriting pharmacie/src/main/java/ma/ensa/pharmacie/repository/CategorieRepository.java


In [11]:
%%writefile pharmacie/src/main/java/ma/ensa/pharmacie/repository/FournisseurRepository.java
package ma.ensa.pharmacie.repository;

import ma.ensa.pharmacie.entity.Fournisseur;
import org.springframework.data.jpa.repository.JpaRepository;

public interface FournisseurRepository extends JpaRepository<Fournisseur, Long> {
}

Overwriting pharmacie/src/main/java/ma/ensa/pharmacie/repository/FournisseurRepository.java


In [12]:
%%writefile pharmacie/src/main/java/ma/ensa/pharmacie/repository/TagRepository.java
package ma.ensa.pharmacie.repository;

import ma.ensa.pharmacie.entity.Tag;
import org.springframework.data.jpa.repository.JpaRepository;

public interface TagRepository extends JpaRepository<Tag, Long> {
}

Overwriting pharmacie/src/main/java/ma/ensa/pharmacie/repository/TagRepository.java


In [13]:
%%writefile pharmacie/src/main/java/ma/ensa/pharmacie/repository/ProduitRepository.java
package ma.ensa.pharmacie.repository;

import ma.ensa.pharmacie.entity.Produit;
import org.springframework.data.jpa.repository.JpaRepository;

public interface ProduitRepository extends JpaRepository<Produit, Long> {
}

Overwriting pharmacie/src/main/java/ma/ensa/pharmacie/repository/ProduitRepository.java


## 7. Service produit

Dans les JSON de test, `categorie`, `fournisseur` et `tags` ne contiennent que des `id`. Si on les passait tels quels à `save()`, la réponse afficherait des objets vides (`nom: null`...). Donc le service va rechercher les vrais objets en base. Si un id n'existe pas, on renvoie **400 Bad Request** avec un message clair.

Pour le `PUT`, on modifie le détail existant au lieu d'en créer un nouveau, comme ça on ne laisse pas de lignes orphelines dans `detail_produit`.

In [14]:
%%writefile pharmacie/src/main/java/ma/ensa/pharmacie/service/ProduitService.java
package ma.ensa.pharmacie.service;

import lombok.RequiredArgsConstructor;
import ma.ensa.pharmacie.entity.*;
import ma.ensa.pharmacie.repository.*;
import org.springframework.http.HttpStatus;
import org.springframework.stereotype.Service;
import org.springframework.transaction.annotation.Transactional;
import org.springframework.web.server.ResponseStatusException;

import java.util.HashSet;
import java.util.List;
import java.util.Optional;
import java.util.Set;

@Service
@RequiredArgsConstructor
public class ProduitService {

    private final ProduitRepository produitRepository;
    private final CategorieRepository categorieRepository;
    private final FournisseurRepository fournisseurRepository;
    private final TagRepository tagRepository;

    public List<Produit> findAll() {
        return produitRepository.findAll();
    }

    public Optional<Produit> findById(Long id) {
        return produitRepository.findById(id);
    }

    @Transactional
    public Produit create(Produit donnees) {
        Produit produit = new Produit();
        remplir(produit, donnees);
        return produitRepository.save(produit);
    }

    @Transactional
    public Optional<Produit> update(Long id, Produit donnees) {
        return produitRepository.findById(id).map(existant -> {
            remplir(existant, donnees);
            return produitRepository.save(existant);
        });
    }

    @Transactional
    public boolean delete(Long id) {
        if (!produitRepository.existsById(id)) {
            return false;
        }
        produitRepository.deleteById(id);
        return true;
    }

    // copie les champs reçus dans l'objet cible et résout les relations par leur id
    private void remplir(Produit cible, Produit source) {
        cible.setNom(source.getNom());
        cible.setDescription(source.getDescription());
        cible.setPrix(source.getPrix());
        cible.setCodeBarre(source.getCodeBarre());
        cible.setDateExpiration(source.getDateExpiration());

        // ManyToOne : catégorie
        if (source.getCategorie() != null && source.getCategorie().getId() != null) {
            Long catId = source.getCategorie().getId();
            cible.setCategorie(categorieRepository.findById(catId)
                    .orElseThrow(() -> new ResponseStatusException(
                            HttpStatus.BAD_REQUEST, "Catégorie introuvable : " + catId)));
        } else {
            cible.setCategorie(null);
        }

        // ManyToOne : fournisseur
        if (source.getFournisseur() != null && source.getFournisseur().getId() != null) {
            Long fId = source.getFournisseur().getId();
            cible.setFournisseur(fournisseurRepository.findById(fId)
                    .orElseThrow(() -> new ResponseStatusException(
                            HttpStatus.BAD_REQUEST, "Fournisseur introuvable : " + fId)));
        } else {
            cible.setFournisseur(null);
        }

        // OneToOne : on met à jour le détail existant s'il y en a un
        DetailProduit d = source.getDetailProduit();
        if (d == null) {
            cible.setDetailProduit(null);
        } else {
            DetailProduit detail = cible.getDetailProduit() != null
                    ? cible.getDetailProduit() : new DetailProduit();
            detail.setFabricant(d.getFabricant());
            detail.setPaysOrigine(d.getPaysOrigine());
            detail.setComposition(d.getComposition());
            detail.setDosage(d.getDosage());
            detail.setIndications(d.getIndications());
            cible.setDetailProduit(detail);
        }

        // ManyToMany : tags
        Set<Tag> tags = new HashSet<>();
        if (source.getTags() != null) {
            for (Tag t : source.getTags()) {
                if (t.getId() == null) {
                    throw new ResponseStatusException(HttpStatus.BAD_REQUEST, "Un tag doit avoir un id");
                }
                tags.add(tagRepository.findById(t.getId())
                        .orElseThrow(() -> new ResponseStatusException(
                                HttpStatus.BAD_REQUEST, "Tag introuvable : " + t.getId())));
            }
        }
        cible.getTags().clear();
        cible.getTags().addAll(tags);
    }
}

Overwriting pharmacie/src/main/java/ma/ensa/pharmacie/service/ProduitService.java


## 8. Contrôleurs REST (`/api/...`)

Codes HTTP utilisés : `200` lecture/modification, `201` création, `204` suppression, `404` id inconnu.

In [15]:
%%writefile pharmacie/src/main/java/ma/ensa/pharmacie/controller/ProduitRestController.java
package ma.ensa.pharmacie.controller;

import lombok.RequiredArgsConstructor;
import ma.ensa.pharmacie.entity.Produit;
import ma.ensa.pharmacie.service.ProduitService;
import org.springframework.http.ResponseEntity;
import org.springframework.web.bind.annotation.*;

import java.net.URI;
import java.util.List;

@RestController
@RequestMapping("/api/produits")
@RequiredArgsConstructor
public class ProduitRestController {

    private final ProduitService service;

    @GetMapping
    public List<Produit> lister() {
        return service.findAll();
    }

    @GetMapping("/{id}")
    public ResponseEntity<Produit> afficher(@PathVariable Long id) {
        return service.findById(id)
                .map(ResponseEntity::ok)
                .orElse(ResponseEntity.notFound().build());
    }

    @PostMapping
    public ResponseEntity<Produit> ajouter(@RequestBody Produit produit) {
        Produit cree = service.create(produit);
        return ResponseEntity.created(URI.create("/api/produits/" + cree.getId())).body(cree);
    }

    @PutMapping("/{id}")
    public ResponseEntity<Produit> modifier(@PathVariable Long id, @RequestBody Produit produit) {
        return service.update(id, produit)
                .map(ResponseEntity::ok)
                .orElse(ResponseEntity.notFound().build());
    }

    @DeleteMapping("/{id}")
    public ResponseEntity<Void> supprimer(@PathVariable Long id) {
        return service.delete(id)
                ? ResponseEntity.noContent().build()
                : ResponseEntity.notFound().build();
    }
}

Overwriting pharmacie/src/main/java/ma/ensa/pharmacie/controller/ProduitRestController.java


In [16]:
%%writefile pharmacie/src/main/java/ma/ensa/pharmacie/controller/CategorieRestController.java
package ma.ensa.pharmacie.controller;

import lombok.RequiredArgsConstructor;
import ma.ensa.pharmacie.entity.Categorie;
import ma.ensa.pharmacie.repository.CategorieRepository;
import org.springframework.http.ResponseEntity;
import org.springframework.web.bind.annotation.*;

import java.net.URI;
import java.util.List;

@RestController
@RequestMapping("/api/categories")
@RequiredArgsConstructor
public class CategorieRestController {

    private final CategorieRepository repository;

    @GetMapping
    public List<Categorie> lister() {
        return repository.findAll();
    }

    @GetMapping("/{id}")
    public ResponseEntity<Categorie> afficher(@PathVariable Long id) {
        return repository.findById(id)
                .map(ResponseEntity::ok)
                .orElse(ResponseEntity.notFound().build());
    }

    @PostMapping
    public ResponseEntity<Categorie> ajouter(@RequestBody Categorie categorie) {
        categorie.setId(null); // on laisse MySQL générer l'id
        Categorie cree = repository.save(categorie);
        return ResponseEntity.created(URI.create("/api/categories/" + cree.getId())).body(cree);
    }

    @PutMapping("/{id}")
    public ResponseEntity<Categorie> modifier(@PathVariable Long id, @RequestBody Categorie donnees) {
        return repository.findById(id).map(c -> {
            c.setNom(donnees.getNom());
            c.setDescription(donnees.getDescription());
            return ResponseEntity.ok(repository.save(c));
        }).orElse(ResponseEntity.notFound().build());
    }

    @DeleteMapping("/{id}")
    public ResponseEntity<Void> supprimer(@PathVariable Long id) {
        if (!repository.existsById(id)) {
            return ResponseEntity.notFound().build();
        }
        repository.deleteById(id);
        return ResponseEntity.noContent().build();
    }
}

Overwriting pharmacie/src/main/java/ma/ensa/pharmacie/controller/CategorieRestController.java


In [17]:
%%writefile pharmacie/src/main/java/ma/ensa/pharmacie/controller/FournisseurRestController.java
package ma.ensa.pharmacie.controller;

import lombok.RequiredArgsConstructor;
import ma.ensa.pharmacie.entity.Fournisseur;
import ma.ensa.pharmacie.repository.FournisseurRepository;
import org.springframework.http.ResponseEntity;
import org.springframework.web.bind.annotation.*;

import java.net.URI;
import java.util.List;

@RestController
@RequestMapping("/api/fournisseurs")
@RequiredArgsConstructor
public class FournisseurRestController {

    private final FournisseurRepository repository;

    @GetMapping
    public List<Fournisseur> lister() {
        return repository.findAll();
    }

    @GetMapping("/{id}")
    public ResponseEntity<Fournisseur> afficher(@PathVariable Long id) {
        return repository.findById(id)
                .map(ResponseEntity::ok)
                .orElse(ResponseEntity.notFound().build());
    }

    @PostMapping
    public ResponseEntity<Fournisseur> ajouter(@RequestBody Fournisseur fournisseur) {
        fournisseur.setId(null);
        Fournisseur cree = repository.save(fournisseur);
        return ResponseEntity.created(URI.create("/api/fournisseurs/" + cree.getId())).body(cree);
    }

    @PutMapping("/{id}")
    public ResponseEntity<Fournisseur> modifier(@PathVariable Long id, @RequestBody Fournisseur donnees) {
        return repository.findById(id).map(f -> {
            f.setNom(donnees.getNom());
            f.setContact(donnees.getContact());
            f.setAdresse(donnees.getAdresse());
            return ResponseEntity.ok(repository.save(f));
        }).orElse(ResponseEntity.notFound().build());
    }

    @DeleteMapping("/{id}")
    public ResponseEntity<Void> supprimer(@PathVariable Long id) {
        if (!repository.existsById(id)) {
            return ResponseEntity.notFound().build();
        }
        repository.deleteById(id);
        return ResponseEntity.noContent().build();
    }
}

Overwriting pharmacie/src/main/java/ma/ensa/pharmacie/controller/FournisseurRestController.java


In [18]:
%%writefile pharmacie/src/main/java/ma/ensa/pharmacie/controller/TagRestController.java
package ma.ensa.pharmacie.controller;

import lombok.RequiredArgsConstructor;
import ma.ensa.pharmacie.entity.Tag;
import ma.ensa.pharmacie.repository.TagRepository;
import org.springframework.http.ResponseEntity;
import org.springframework.web.bind.annotation.*;

import java.net.URI;
import java.util.List;

@RestController
@RequestMapping("/api/tags")
@RequiredArgsConstructor
public class TagRestController {

    private final TagRepository repository;

    @GetMapping
    public List<Tag> lister() {
        return repository.findAll();
    }

    @GetMapping("/{id}")
    public ResponseEntity<Tag> afficher(@PathVariable Long id) {
        return repository.findById(id)
                .map(ResponseEntity::ok)
                .orElse(ResponseEntity.notFound().build());
    }

    @PostMapping
    public ResponseEntity<Tag> ajouter(@RequestBody Tag tag) {
        tag.setId(null);
        Tag cree = repository.save(tag);
        return ResponseEntity.created(URI.create("/api/tags/" + cree.getId())).body(cree);
    }

    @PutMapping("/{id}")
    public ResponseEntity<Tag> modifier(@PathVariable Long id, @RequestBody Tag donnees) {
        return repository.findById(id).map(t -> {
            t.setLibelle(donnees.getLibelle());
            return ResponseEntity.ok(repository.save(t));
        }).orElse(ResponseEntity.notFound().build());
    }

    @DeleteMapping("/{id}")
    public ResponseEntity<Void> supprimer(@PathVariable Long id) {
        if (!repository.existsById(id)) {
            return ResponseEntity.notFound().build();
        }
        repository.deleteById(id);
        return ResponseEntity.noContent().build();
    }
}

Overwriting pharmacie/src/main/java/ma/ensa/pharmacie/controller/TagRestController.java


## 9. Contrôleurs MVC + Thymeleaf

Pages HTML : `/produits`, `/produits/{id}`, `/categories`, `/fournisseurs`, `/tags`. Les produits dont la date d'expiration est dépassée sont affichés en rouge.

In [19]:
%%writefile pharmacie/src/main/java/ma/ensa/pharmacie/controller/ProduitViewController.java
package ma.ensa.pharmacie.controller;

import lombok.RequiredArgsConstructor;
import ma.ensa.pharmacie.entity.Produit;
import ma.ensa.pharmacie.service.ProduitService;
import org.springframework.http.HttpStatus;
import org.springframework.stereotype.Controller;
import org.springframework.ui.Model;
import org.springframework.web.bind.annotation.GetMapping;
import org.springframework.web.bind.annotation.PathVariable;
import org.springframework.web.server.ResponseStatusException;

import java.time.LocalDate;

@Controller
@RequiredArgsConstructor
public class ProduitViewController {

    private final ProduitService service;

    @GetMapping("/")
    public String accueil() {
        return "redirect:/produits";
    }

    @GetMapping("/produits")
    public String liste(Model model) {
        model.addAttribute("produits", service.findAll());
        model.addAttribute("today", LocalDate.now());
        return "produits";
    }

    @GetMapping("/produits/{id}")
    public String detail(@PathVariable Long id, Model model) {
        Produit p = service.findById(id)
                .orElseThrow(() -> new ResponseStatusException(HttpStatus.NOT_FOUND, "Produit introuvable"));
        model.addAttribute("produit", p);
        return "produit-detail";
    }
}

Overwriting pharmacie/src/main/java/ma/ensa/pharmacie/controller/ProduitViewController.java


In [20]:
%%writefile pharmacie/src/main/java/ma/ensa/pharmacie/controller/CategorieViewController.java
package ma.ensa.pharmacie.controller;

import lombok.RequiredArgsConstructor;
import ma.ensa.pharmacie.repository.CategorieRepository;
import org.springframework.stereotype.Controller;
import org.springframework.ui.Model;
import org.springframework.web.bind.annotation.GetMapping;

@Controller
@RequiredArgsConstructor
public class CategorieViewController {

    private final CategorieRepository repository;

    @GetMapping("/categories")
    public String liste(Model model) {
        model.addAttribute("categories", repository.findAll());
        return "categories";
    }
}

Overwriting pharmacie/src/main/java/ma/ensa/pharmacie/controller/CategorieViewController.java


In [21]:
%%writefile pharmacie/src/main/java/ma/ensa/pharmacie/controller/FournisseurViewController.java
package ma.ensa.pharmacie.controller;

import lombok.RequiredArgsConstructor;
import ma.ensa.pharmacie.repository.FournisseurRepository;
import org.springframework.stereotype.Controller;
import org.springframework.ui.Model;
import org.springframework.web.bind.annotation.GetMapping;

@Controller
@RequiredArgsConstructor
public class FournisseurViewController {

    private final FournisseurRepository repository;

    @GetMapping("/fournisseurs")
    public String liste(Model model) {
        model.addAttribute("fournisseurs", repository.findAll());
        return "fournisseurs";
    }
}

Overwriting pharmacie/src/main/java/ma/ensa/pharmacie/controller/FournisseurViewController.java


In [22]:
%%writefile pharmacie/src/main/java/ma/ensa/pharmacie/controller/TagViewController.java
package ma.ensa.pharmacie.controller;

import lombok.RequiredArgsConstructor;
import ma.ensa.pharmacie.repository.TagRepository;
import org.springframework.stereotype.Controller;
import org.springframework.ui.Model;
import org.springframework.web.bind.annotation.GetMapping;

@Controller
@RequiredArgsConstructor
public class TagViewController {

    private final TagRepository repository;

    @GetMapping("/tags")
    public String liste(Model model) {
        model.addAttribute("tags", repository.findAll());
        return "tags";
    }
}

Overwriting pharmacie/src/main/java/ma/ensa/pharmacie/controller/TagViewController.java


In [23]:
%%writefile pharmacie/src/main/resources/templates/layout.html
<!DOCTYPE html>
<html xmlns:th="http://www.thymeleaf.org">
<head th:fragment="head(titre)">
    <meta charset="UTF-8">
    <title th:text="${titre} + ' - Pharmacie'">Pharmacie</title>
    <style>
        body { font-family: Arial, sans-serif; margin: 0; background: #f4f6f8; color: #222; }
        nav { background: #1f6f5c; padding: 12px 24px; }
        nav a { color: white; margin-right: 20px; text-decoration: none; font-weight: bold; }
        main { padding: 24px; }
        table { border-collapse: collapse; width: 100%; background: white; }
        th, td { border: 1px solid #ddd; padding: 8px 10px; text-align: left; }
        th { background: #e3efec; }
        .expire { color: #c0392b; font-weight: bold; }
        .tag { background: #d6eaf8; border-radius: 10px; padding: 2px 8px; margin-right: 4px; font-size: 12px; }
    </style>
</head>
<body>
<nav th:fragment="nav">
    <a th:href="@{/produits}">Produits</a>
    <a th:href="@{/categories}">Catégories</a>
    <a th:href="@{/fournisseurs}">Fournisseurs</a>
    <a th:href="@{/tags}">Tags</a>
</nav>
</body>
</html>

Overwriting pharmacie/src/main/resources/templates/layout.html


In [24]:
%%writefile pharmacie/src/main/resources/templates/produits.html
<!DOCTYPE html>
<html xmlns:th="http://www.thymeleaf.org">
<head th:replace="~{layout :: head('Produits')}"></head>
<body>
<nav th:replace="~{layout :: nav}"></nav>
<main>
    <h1>Stock de la pharmacie</h1>
    <table>
        <thead>
        <tr>
            <th>Id</th><th>Nom</th><th>Prix (DH)</th><th>Code-barres</th>
            <th>Expiration</th><th>Catégorie</th><th>Fournisseur</th><th>Tags</th>
        </tr>
        </thead>
        <tbody>
        <tr th:each="p : ${produits}">
            <td th:text="${p.id}"></td>
            <td><a th:href="@{/produits/{id}(id=${p.id})}" th:text="${p.nom}"></a></td>
            <td th:text="${p.prix}"></td>
            <td th:text="${p.codeBarre}"></td>
            <td th:text="${p.dateExpiration}"
                th:classappend="${p.dateExpiration != null and p.dateExpiration.isBefore(today)} ? 'expire'"></td>
            <td th:text="${p.categorie != null} ? ${p.categorie.nom} : '-'"></td>
            <td th:text="${p.fournisseur != null} ? ${p.fournisseur.nom} : '-'"></td>
            <td><span class="tag" th:each="t : ${p.tags}" th:text="${t.libelle}"></span></td>
        </tr>
        </tbody>
    </table>
</main>
</body>
</html>

Overwriting pharmacie/src/main/resources/templates/produits.html


In [25]:
%%writefile pharmacie/src/main/resources/templates/produit-detail.html
<!DOCTYPE html>
<html xmlns:th="http://www.thymeleaf.org">
<head th:replace="~{layout :: head('Détail produit')}"></head>
<body>
<nav th:replace="~{layout :: nav}"></nav>
<main>
    <h1 th:text="${produit.nom}">Produit</h1>
    <p th:text="${produit.description}"></p>
    <p><b>Prix :</b> <span th:text="${produit.prix}"></span> DH</p>
    <p><b>Code-barres :</b> <span th:text="${produit.codeBarre}"></span></p>
    <p><b>Expiration :</b> <span th:text="${produit.dateExpiration}"></span></p>
    <p><b>Catégorie :</b> <span th:text="${produit.categorie != null} ? ${produit.categorie.nom} : '-'"></span></p>
    <p><b>Fournisseur :</b> <span th:text="${produit.fournisseur != null} ? ${produit.fournisseur.nom} : '-'"></span></p>

    <h2>Détails</h2>
    <div th:if="${produit.detailProduit != null}">
        <p><b>Fabricant :</b> <span th:text="${produit.detailProduit.fabricant}"></span></p>
        <p><b>Pays d'origine :</b> <span th:text="${produit.detailProduit.paysOrigine}"></span></p>
        <p><b>Composition :</b> <span th:text="${produit.detailProduit.composition}"></span></p>
        <p><b>Dosage :</b> <span th:text="${produit.detailProduit.dosage}"></span></p>
        <p><b>Indications :</b> <span th:text="${produit.detailProduit.indications}"></span></p>
    </div>

    <p><b>Tags :</b> <span class="tag" th:each="t : ${produit.tags}" th:text="${t.libelle}"></span></p>
    <p><a th:href="@{/produits}">&larr; Retour à la liste</a></p>
</main>
</body>
</html>

Overwriting pharmacie/src/main/resources/templates/produit-detail.html


In [26]:
%%writefile pharmacie/src/main/resources/templates/categories.html
<!DOCTYPE html>
<html xmlns:th="http://www.thymeleaf.org">
<head th:replace="~{layout :: head('Catégories')}"></head>
<body>
<nav th:replace="~{layout :: nav}"></nav>
<main>
    <h1>Catégories</h1>
    <table>
        <thead><tr><th>Id</th><th>Nom</th><th>Description</th></tr></thead>
        <tbody>
        <tr th:each="c : ${categories}">
            <td th:text="${c.id}"></td>
            <td th:text="${c.nom}"></td>
            <td th:text="${c.description}"></td>
        </tr>
        </tbody>
    </table>
</main>
</body>
</html>

Overwriting pharmacie/src/main/resources/templates/categories.html


In [27]:
%%writefile pharmacie/src/main/resources/templates/fournisseurs.html
<!DOCTYPE html>
<html xmlns:th="http://www.thymeleaf.org">
<head th:replace="~{layout :: head('Fournisseurs')}"></head>
<body>
<nav th:replace="~{layout :: nav}"></nav>
<main>
    <h1>Fournisseurs</h1>
    <table>
        <thead><tr><th>Id</th><th>Nom</th><th>Contact</th><th>Adresse</th></tr></thead>
        <tbody>
        <tr th:each="f : ${fournisseurs}">
            <td th:text="${f.id}"></td>
            <td th:text="${f.nom}"></td>
            <td th:text="${f.contact}"></td>
            <td th:text="${f.adresse}"></td>
        </tr>
        </tbody>
    </table>
</main>
</body>
</html>

Overwriting pharmacie/src/main/resources/templates/fournisseurs.html


In [28]:
%%writefile pharmacie/src/main/resources/templates/tags.html
<!DOCTYPE html>
<html xmlns:th="http://www.thymeleaf.org">
<head th:replace="~{layout :: head('Tags')}"></head>
<body>
<nav th:replace="~{layout :: nav}"></nav>
<main>
    <h1>Tags</h1>
    <table>
        <thead><tr><th>Id</th><th>Libellé</th></tr></thead>
        <tbody>
        <tr th:each="t : ${tags}">
            <td th:text="${t.id}"></td>
            <td th:text="${t.libelle}"></td>
        </tr>
        </tbody>
    </table>
</main>
</body>
</html>

Overwriting pharmacie/src/main/resources/templates/tags.html


## 10. Jeu de données (`data.sql`)

C'est le jeu de données de l'énoncé, adapté aux noms de colonnes générés par Hibernate (`detail_produit`, `pays_origine`). J'utilise `INSERT IGNORE` pour que l'application puisse redémarrer sans erreur de clé dupliquée.

In [29]:
%%writefile pharmacie/src/main/resources/data.sql
INSERT IGNORE INTO categorie (id, nom, description) VALUES
(1, 'Antibiotiques', 'Médicaments utilisés pour traiter les infections bactériennes.'),
(2, 'Antalgiques', 'Médicaments pour soulager la douleur.'),
(3, 'Vitamines', 'Compléments nutritionnels et fortifiants.'),
(4, 'Antiseptiques', 'Produits désinfectants pour usage externe.');

INSERT IGNORE INTO fournisseur (id, nom, contact, adresse) VALUES
(1, 'PharmaMaroc', 'contact@pharmamaroc.ma', 'Casablanca, Maroc'),
(2, 'Medisup France', 'info@medisup.fr', 'Paris, France'),
(3, 'Sandoz', 'export@sandoz.com', 'Bâle, Suisse'),
(4, 'Cooper Pharma', 'support@cooperpharma.ma', 'Rabat, Maroc');

INSERT IGNORE INTO detail_produit (id, fabricant, pays_origine, composition, dosage, indications) VALUES
(1, 'Sanofi', 'France', 'Amoxicilline trihydratée 500 mg', '500 mg', 'Infections ORL et respiratoires'),
(2, 'Bayer', 'Allemagne', 'Paracétamol 500 mg', '500 mg', 'Douleurs légères à modérées, fièvre'),
(3, 'Pfizer', 'États-Unis', 'Vitamine C 1000 mg', '1000 mg', 'Renforcement immunitaire'),
(4, 'Johnson & Johnson', 'Maroc', 'Povidone iodée 10%', '10%', 'Désinfection des plaies'),
(5, 'GSK', 'Royaume-Uni', 'Ibuprofène 400 mg', '400 mg', 'Inflammations et douleurs musculaires');

INSERT IGNORE INTO tag (id, libelle) VALUES
(1, 'Sans ordonnance'),
(2, 'Prescription obligatoire'),
(3, 'En promotion'),
(4, 'Stock limité'),
(5, 'Importé');

INSERT IGNORE INTO produit (id, nom, description, prix, code_barre, date_expiration, categorie_id, fournisseur_id, detail_id) VALUES
(1, 'Amoxicilline 500mg', 'Antibiotique à large spectre.', 45.00, 'AMX500-2025', '2026-03-10', 1, 1, 1),
(2, 'Doliprane 500mg', 'Antalgique et antipyrétique.', 25.50, 'DLP500-2025', '2025-11-01', 2, 2, 2),
(3, 'Vitamine C 1000mg', 'Complément pour renforcer les défenses.', 30.00, 'VITC1000-2025', '2026-06-15', 3, 3, 3),
(4, 'Bétadine Solution 10%', 'Désinfectant pour usage externe.', 18.00, 'BETADINE10-2025', '2027-01-30', 4, 4, 4),
(5, 'Nurofen 400mg', 'Soulage la douleur et la fièvre.', 42.00, 'NURO400-2025', '2026-02-20', 2, 3, 5);

INSERT IGNORE INTO produit_tag (produit_id, tag_id) VALUES
(1, 2),
(2, 1),
(2, 3),
(3, 1),
(3, 5),
(4, 1),
(4, 4),
(5, 1),
(5, 5);

Overwriting pharmacie/src/main/resources/data.sql


## 11. Lancer l'application

La cellule démarre `mvn spring-boot:run` en arrière-plan (les logs vont dans `pharmacie/app.log`) et attend que l'API réponde. Le premier démarrage peut durer 1 à 3 minutes (téléchargement des dépendances Maven).

In [30]:
import subprocess, time, requests, shutil

BASE_URL = "http://localhost:8080"

mvn = shutil.which("mvn")          # sous Windows ça trouve mvn.cmd
assert mvn, "Maven introuvable : vérifier le PATH puis redémarrer Jupyter"

log = open("pharmacie/app.log", "w")
serveur = subprocess.Popen([mvn, "-q", "spring-boot:run"], cwd="pharmacie",
                           stdout=log, stderr=subprocess.STDOUT)

demarre = False
for _ in range(120):                      # 120 x 3 s = 6 min max
    if serveur.poll() is not None:
        break                             # maven s'est arrêté : voir app.log
    try:
        if requests.get(BASE_URL + "/api/categories", timeout=2).status_code == 200:
            demarre = True
            break
    except requests.exceptions.RequestException:
        pass
    time.sleep(3)

print("Application démarrée" if demarre else "Échec du démarrage, regarde pharmacie/app.log")

Application démarrée


## 12. Tests des routes

Ce sont les mêmes tests que ceux demandés avec Postman, mais écrits avec `requests` pour qu'on voie directement les résultats dans le notebook. La cellule suivante définit les corps JSON de l'énoncé.

> **Remarque :** le JSON de l'énoncé pour `POST /api/produits` utilise `categorie.id = 5` et `fournisseur.id = 5`, alors que le jeu de données n'a que 4 catégories et 4 fournisseurs. Il faut donc d'abord ajouter la catégorie « Anti-inflammatoires » et le fournisseur « Novartis » (ils reçoivent l'id 5) avant d'ajouter Voltaren. C'est pour ça que les `POST` categories et fournisseurs passent en premier. Dans le test Python je récupère l'id renvoyé au lieu de supposer `5`.

In [31]:
import json

nouvelle_categorie = {
    "nom": "Anti-inflammatoires",
    "description": "Médicaments pour réduire l'inflammation et la douleur."
}
nouveau_fournisseur = {
    "nom": "Novartis",
    "contact": "contact@novartis.com",
    "adresse": "Genève, Suisse"
}

def voltaren(categorie_id, fournisseur_id):
    return {
        "nom": "Voltaren 50mg",
        "description": "Anti-inflammatoire pour douleurs musculaires et articulaires.",
        "prix": 58.90,
        "codeBarre": "VOLTA50-2025",
        "dateExpiration": "2026-12-15",
        "categorie": {"id": categorie_id},
        "fournisseur": {"id": fournisseur_id},
        "detailProduit": {
            "fabricant": "Novartis",
            "paysOrigine": "Suisse",
            "composition": "Diclofénac sodique 50 mg",
            "dosage": "50 mg",
            "indications": "Traitement des douleurs musculaires"
        },
        "tags": [{"id": 1}, {"id": 3}]
    }

modif_produit_2 = {
    "nom": "Doliprane 500mg (Nouveau)",
    "description": "Antalgique et antipyrétique — nouvelle formule 2025.",
    "prix": 27.00,
    "codeBarre": "DLP500-NEW",
    "dateExpiration": "2026-05-10",
    "categorie": {"id": 2},
    "fournisseur": {"id": 2},
    "detailProduit": {
        "fabricant": "Bayer",
        "paysOrigine": "Allemagne",
        "composition": "Paracétamol 500 mg",
        "dosage": "500 mg",
        "indications": "Douleur et fièvre légère"
    },
    "tags": [{"id": 1}, {"id": 3}]
}
modif_fournisseur_3 = {
    "nom": "Sandoz Pharma",
    "contact": "contact@sandoz.com",
    "adresse": "Zürich, Suisse"
}
modif_categorie_2 = {
    "nom": "Antalgiques & Anti-douleurs",
    "description": "Catégorie regroupant tous les médicaments contre la douleur."
}
print("Corps JSON prêts")

Corps JSON prêts


In [32]:
def appel(methode, chemin, body=None, attendu=200):
    """Envoie la requête, vérifie le code HTTP et affiche le résultat."""
    r = requests.request(methode, BASE_URL + chemin, json=body, timeout=15)
    ok = "OK  " if r.status_code == attendu else "ERREUR"
    print(f"[{ok}] {methode:6} {chemin:25} -> {r.status_code} (attendu {attendu})")
    assert r.status_code == attendu, r.text
    return r

# --- Lecture ---
r = appel("GET", "/api/produits")
print("   nombre de produits :", len(r.json()))
r = appel("GET", "/api/produits/1")
print("  ", json.dumps(r.json(), ensure_ascii=False, indent=2))
for chemin in ["/api/categories", "/api/fournisseurs", "/api/tags"]:
    r = appel("GET", chemin)
    print("   ", chemin, ":", len(r.json()), "éléments")
appel("GET", "/api/produits/999", attendu=404)

[OK  ] GET    /api/produits             -> 200 (attendu 200)
   nombre de produits : 5
[OK  ] GET    /api/produits/1           -> 200 (attendu 200)
   {
  "id": 1,
  "nom": "Amoxicilline 500mg",
  "description": "Antibiotique à large spectre.",
  "prix": 45.0,
  "codeBarre": "AMX500-2025",
  "dateExpiration": "2026-03-10",
  "categorie": {
    "id": 1,
    "nom": "Antibiotiques",
    "description": "Médicaments utilisés pour traiter les infections bactériennes."
  },
  "fournisseur": {
    "id": 1,
    "nom": "PharmaMaroc",
    "contact": "contact@pharmamaroc.ma",
    "adresse": "Casablanca, Maroc"
  },
  "detailProduit": {
    "id": 1,
    "fabricant": "Sanofi",
    "paysOrigine": "France",
    "composition": "Amoxicilline trihydratée 500 mg",
    "dosage": "500 mg",
    "indications": "Infections ORL et respiratoires"
  },
  "tags": [
    {
      "id": 2,
      "libelle": "Prescription obligatoire"
    }
  ]
}
[OK  ] GET    /api/categories           -> 200 (attendu 200)
    /api/cate

<Response [404]>

In [33]:
# --- Ajout (POST) : catégorie et fournisseur d'abord, puis le produit ---
r = appel("POST", "/api/categories", nouvelle_categorie, attendu=201)
cat_id = r.json()["id"]
r = appel("POST", "/api/fournisseurs", nouveau_fournisseur, attendu=201)
four_id = r.json()["id"]
print("   ids créés : catégorie", cat_id, "/ fournisseur", four_id)

r = appel("POST", "/api/produits", voltaren(cat_id, four_id), attendu=201)
produit_cree = r.json()
print(json.dumps(produit_cree, ensure_ascii=False, indent=2))
id_voltaren = produit_cree["id"]

[OK  ] POST   /api/categories           -> 201 (attendu 201)
[OK  ] POST   /api/fournisseurs         -> 201 (attendu 201)
   ids créés : catégorie 9 / fournisseur 9
[OK  ] POST   /api/produits             -> 201 (attendu 201)
{
  "id": 10,
  "nom": "Voltaren 50mg",
  "description": "Anti-inflammatoire pour douleurs musculaires et articulaires.",
  "prix": 58.9,
  "codeBarre": "VOLTA50-2025",
  "dateExpiration": "2026-12-15",
  "categorie": {
    "id": 9,
    "nom": "Anti-inflammatoires",
    "description": "Médicaments pour réduire l'inflammation et la douleur."
  },
  "fournisseur": {
    "id": 9,
    "nom": "Novartis",
    "contact": "contact@novartis.com",
    "adresse": "Genève, Suisse"
  },
  "detailProduit": {
    "id": 10,
    "fabricant": "Novartis",
    "paysOrigine": "Suisse",
    "composition": "Diclofénac sodique 50 mg",
    "dosage": "50 mg",
    "indications": "Traitement des douleurs musculaires"
  },
  "tags": [
    {
      "id": 1,
      "libelle": "Sans ordonnance"
  

In [34]:
# --- Modification (PUT) ---
r = appel("PUT", "/api/produits/2", modif_produit_2)
print("   Doliprane après modification :", r.json()["nom"], "|", r.json()["prix"], "DH")
r = appel("PUT", "/api/fournisseurs/3", modif_fournisseur_3)
print("   fournisseur 3 :", r.json()["nom"], "|", r.json()["adresse"])
r = appel("PUT", "/api/categories/2", modif_categorie_2)
print("   catégorie 2 :", r.json()["nom"])
appel("PUT", "/api/produits/999", modif_produit_2, attendu=404)

[OK  ] PUT    /api/produits/2           -> 200 (attendu 200)
   Doliprane après modification : Doliprane 500mg (Nouveau) | 27.0 DH
[OK  ] PUT    /api/fournisseurs/3       -> 200 (attendu 200)
   fournisseur 3 : Sandoz Pharma | Zürich, Suisse
[OK  ] PUT    /api/categories/2         -> 200 (attendu 200)
   catégorie 2 : Antalgiques & Anti-douleurs
[OK  ] PUT    /api/produits/999         -> 404 (attendu 404)


<Response [404]>

In [35]:
# --- Suppression (DELETE) ---
appel("DELETE", f"/api/produits/{id_voltaren}", attendu=204)
appel("GET", f"/api/produits/{id_voltaren}", attendu=404)
print("Tous les tests de l'API sont passés")

[OK  ] DELETE /api/produits/10          -> 204 (attendu 204)
[OK  ] GET    /api/produits/10          -> 404 (attendu 404)
Tous les tests de l'API sont passés


In [36]:
# --- Pages Thymeleaf ---
for page in ["/produits", "/produits/1", "/categories", "/fournisseurs", "/tags"]:
    r = requests.get(BASE_URL + page, timeout=15)
    print(f"{page:16} -> {r.status_code}  ({len(r.text)} caractères de HTML)")
    assert r.status_code == 200

print()
print("'Doliprane' présent dans /produits :", "Doliprane" in requests.get(BASE_URL + "/produits").text)

/produits        -> 200  (3318 caractères de HTML)
/produits/1      -> 200  (1875 caractères de HTML)
/categories      -> 200  (2712 caractères de HTML)
/fournisseurs    -> 200  (2674 caractères de HTML)
/tags            -> 200  (1587 caractères de HTML)

'Doliprane' présent dans /produits : True


## 13. Collection Postman

Pour les tests demandés « avec Postman » : cette cellule génère un fichier à importer (*Import → Upload Files*). Chaque requête contient un test qui vérifie le code HTTP. À lancer **dans l'ordre**, sur la base fraîche (le POST de Voltaren utilise les ids 5 créés juste avant).

In [37]:
H = [{"key": "Content-Type", "value": "application/json"}]

def requete(nom, methode, chemin, body=None, statut=200):
    script = f"pm.test('Statut {statut}', function () {{ pm.response.to.have.status({statut}); }});"
    r = {
        "name": nom,
        "event": [{"listen": "test", "script": {"type": "text/javascript", "exec": [script]}}],
        "request": {
            "method": methode,
            "header": H,
            "url": {"raw": "{{baseUrl}}" + chemin, "host": ["{{baseUrl}}"], "path": chemin.strip("/").split("/")},
        },
    }
    if body is not None:
        r["request"]["body"] = {"mode": "raw", "raw": json.dumps(body, ensure_ascii=False, indent=2)}
    return r

collection = {
    "info": {
        "name": "TP1 - Pharmacie",
        "schema": "https://schema.getpostman.com/json/collection/v2.1.0/collection.json",
    },
    "variable": [{"key": "baseUrl", "value": "http://localhost:8080"}],
    "item": [
        requete("Lister les produits", "GET", "/api/produits"),
        requete("Afficher le produit 1", "GET", "/api/produits/1"),
        requete("Lister les catégories", "GET", "/api/categories"),
        requete("Lister les fournisseurs", "GET", "/api/fournisseurs"),
        requete("Lister les tags", "GET", "/api/tags"),
        requete("Ajouter une catégorie", "POST", "/api/categories", nouvelle_categorie, 201),
        requete("Ajouter un fournisseur", "POST", "/api/fournisseurs", nouveau_fournisseur, 201),
        requete("Ajouter un produit (Voltaren)", "POST", "/api/produits", voltaren(5, 5), 201),
        requete("Modifier le produit 2", "PUT", "/api/produits/2", modif_produit_2),
        requete("Modifier le fournisseur 3", "PUT", "/api/fournisseurs/3", modif_fournisseur_3),
        requete("Modifier la catégorie 2", "PUT", "/api/categories/2", modif_categorie_2),
        requete("Supprimer le produit 6", "DELETE", "/api/produits/6", None, 204),
    ],
}

with open("postman/TP1-pharmacie.postman_collection.json", "w", encoding="utf-8") as f:
    json.dump(collection, f, ensure_ascii=False, indent=2)
print("Collection enregistrée :", len(collection["item"]), "requêtes")

Collection enregistrée : 12 requêtes


## 14. Arrêter l'application

In [38]:
serveur.terminate()
try:
    serveur.wait(timeout=20)
except subprocess.TimeoutExpired:
    serveur.kill()
log.close()
print("Application arrêtée")

Application arrêtée


## Conclusion

- Les 4 types de relations JPA (`OneToOne`, `OneToMany`, `ManyToOne`, `ManyToMany`) sont mappés et la table de jointure `produit_tag` est générée par Hibernate.
- Les données sont accessibles de deux façons : pages Thymeleaf (`/produits`...) et API REST (`/api/...`) avec les codes 200, 201, 204 et 404.
- Les ids inexistants dans un POST/PUT de produit donnent un 400 explicite plutôt qu'une erreur 500.

**Pour publier sur GitHub :** ajouter un `.gitignore` avec `pharmacie/target/` et `pharmacie/app.log`, exécuter toutes les cellules, sauvegarder le notebook (les sorties sont conservées dans le fichier), puis `git add`, `git commit` et `git push`.